# Drowsiness Detection Monitor
This notebook builds a real-time drowsiness detection monitor using YOLO, MediaPipe Tasks API, and OpenCV. It also plays an alert sound when drowsiness is detected and saves output artifacts in the project folder.

In [1]:
import os
import time
import cv2
import numpy as np
from datetime import datetime

try:
    import winsound
except ImportError:
    winsound = None

from ultralytics import YOLO
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.core import base_options
from mediapipe.tasks.python.vision.core import image as mp_image
from mediapipe.tasks.python.vision.core import vision_task_running_mode as vtrm

In [2]:
# Configure project folder and model paths
project_folder = r"C:\Aniket\AI Course\repository\repository\drowsinessDetection"
os.makedirs(project_folder, exist_ok=True)

face_model_path = os.path.join(project_folder, "face_landmarker.task")
if not os.path.exists(face_model_path):
    raise FileNotFoundError(f"Could not find face_landmarker.task at {face_model_path}")

# Optionally, use a lightweight YOLO face model or a custom eye detector.
yolo_model = YOLO("yolov8n.pt")  # Replace with your preferred YOLO model if needed

# Face landmark tracker options
face_options = vision.FaceLandmarkerOptions(
    base_options=base_options.BaseOptions(model_asset_path=face_model_path),
    running_mode=vtrm.VisionTaskRunningMode.VIDEO,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

face_landmarker = vision.FaceLandmarker.create_from_options(face_options)
print("Initialized project folder and models")

Initialized project folder and models


In [3]:
# Define eye aspect ratio and drowsiness detection logic
LEFT_EYE_INDICES = [33, 160, 158, 133, 153, 144]
RIGHT_EYE_INDICES = [263, 387, 385, 362, 380, 373]
EAR_THRESHOLD = 0.25
CONSECUTIVE_FRAMES = 20


def euclidean_distance(point1, point2):
    return np.linalg.norm(np.array(point1) - np.array(point2))


def eye_aspect_ratio(landmarks, eye_indices, image_width, image_height):
    points = [(int(landmarks[i].x * image_width), int(landmarks[i].y * image_height)) for i in eye_indices]
    A = euclidean_distance(points[1], points[5])
    B = euclidean_distance(points[2], points[4])
    C = euclidean_distance(points[0], points[3])
    if C == 0:
        return 0.0
    ear = (A + B) / (2.0 * C)
    return ear


def check_drowsiness(left_ear, right_ear, threshold=EAR_THRESHOLD):
    ear = (left_ear + right_ear) / 2.0
    return ear < threshold, ear

In [ ]:
# Run webcam capture and real-time inference
alert_cooldown = 3.0  # seconds between repeated alerts after one has played
last_alert_time = 0.0
closed_start_time = None


def play_alert_sound():
    if winsound:
        winsound.Beep(2500, 500)
    else:
        print("[ALERT] Drowsiness detected - play sound manually if supported.")


def annotate_frame(frame, text, color=(0, 0, 255)):
    cv2.putText(frame, text, (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)


def detect_drowsiness_from_webcam(camera_index=0, duration=40, close_duration=4.0):
    global last_alert_time, closed_start_time

    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open webcam at index {camera_index}")

    start_time = time.time()

    while True:
        if time.time() - start_time >= duration:
            break

        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        height, width, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_img = mp_image.Image(image_format=mp_image.ImageFormat.SRGB, data=rgb_frame)
        timestamp_ms = int(time.time() * 1000)

        result = face_landmarker.detect_for_video(mp_img, timestamp_ms)
        drowsy = False
        ear = 0.0

        if result.face_landmarks:
            face_landmarks = result.face_landmarks[0]
            left_ear = eye_aspect_ratio(face_landmarks, LEFT_EYE_INDICES, width, height)
            right_ear = eye_aspect_ratio(face_landmarks, RIGHT_EYE_INDICES, width, height)
            drowsy, ear = check_drowsiness(left_ear, right_ear)

            for landmark in face_landmarks:
                x = int(landmark.x * width)
                y = int(landmark.y * height)
                cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

        current_time = time.time()
        if drowsy:
            if closed_start_time is None:
                closed_start_time = current_time

            closed_time = current_time - closed_start_time
            annotate_frame(frame, f"Eyes closed: {closed_time:.1f}s", (0, 0, 255))

            if closed_time >= close_duration:
                if current_time - last_alert_time > alert_cooldown:
                    play_alert_sound()
                    last_alert_time = current_time
                    annotate_frame(frame, f"DROWSY: EAR={ear:.2f}", (0, 0, 255))
        else:
            closed_start_time = None
            annotate_frame(frame, f"Awake: EAR={ear:.2f}", (0, 255, 0))

        cv2.imshow("Drowsiness Detector", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


# Run the detector
# detect_drowsiness_from_webcam(camera_index=0, duration=20, close_duration=2.0)

In [5]:
# Save output artifacts in target folder
log_file = os.path.join(project_folder, f"drowsiness_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
with open(log_file, "w") as f:
    f.write("Drowsiness detector notebook run created artifacts in the target folder.\n")
    f.write(f"Face landmarker model: {face_model_path}\n")
    f.write(f"Created at: {datetime.now()}\n")
print(f"Saved notebook artifact log: {log_file}")

# Recommended usage
print("Run detect_drowsiness_from_webcam(camera_index=0) to start the detector.")

Saved notebook artifact log: C:\Aniket\AI Course\repository\repository\drowsinessDetection\drowsiness_log_20260618_175836.txt
Run detect_drowsiness_from_webcam(camera_index=0) to start the detector.


In [6]:
detect_drowsiness_from_webcam(camera_index=0)